In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PyEMD import EMD

# ==============================================================================
# 1. AUTONOMOUS SIGNAL FRACTURING
# ==============================================================================
# Assume 'raw_french_temp_array' is a 1D numpy array of your unfiltered data
# We instantiate the EMD engine and run the decomposition
emd_engine = EMD()
imfs = emd_engine.emd(raw_french_temp_array)

# The output 'imfs' is a 2D numpy array where each row is an Intrinsic Mode Function.
# The final row is the residual (trend).
num_imfs = imfs.shape[0]
print(f"The EMD algorithm extracted {num_imfs - 1} IMFs and 1 Residual.")

# ==============================================================================
# 2. VISUAL DISSECTION
# ==============================================================================
# Build a stacked plot architecture
fig, axes = plt.subplots(num_imfs, 1, figsize=(12, 2 * num_imfs), sharex=True)
plt.subplots_adjust(hspace=0.4)

time_axis = np.arange(len(raw_french_temp_array))

for i in range(num_imfs):
    ax = axes[i]
    ax.plot(time_axis, imfs[i], color='black', linewidth=1.2)
    
    if i == num_imfs - 1:
        ax.set_title("Residual (Secular Anthropogenic Trend)")
    else:
        ax.set_title(f"IMF {i+1} (Increasing Wavelength)")
        
plt.xlabel("Time (Months)")
plt.show()

In [ ]:
import numpy as np
import pywt
import matplotlib.pyplot as plt

# ==============================================================================
# 1. APPLY THE MORLET WAVELET TRANSFORM
# ==============================================================================
# Assume df_ortho is a pandas DataFrame and EA_Lag0 is your standardized array
signal = df_ortho['EA_Lag0'].values
time_axis = np.arange(len(signal))

# We define the scales (which correlate inversely with frequency/period)
# We calculate up to 200 scales to capture high and low frequencies
scales = np.arange(1, 200)

# Execute the Continuous Wavelet Transform using a Complex Morlet wavelet
# cmor1.5-1.0 indicates a bandwidth of 1.5 and center frequency of 1.0
coefficients, frequencies = pywt.cwt(signal, scales, 'cmor1.5-1.0')

# ==============================================================================
# 2. RENDER THE 3D THERMODYNAMIC HEATMAP
# ==============================================================================
# Calculate the absolute mathematical power of the complex matrix
wave_power = np.abs(coefficients)**2

plt.figure(figsize=(12, 6))
# Render the heatmap. We use pcolormesh for proper scale rendering.
plt.pcolormesh(time_axis, frequencies, wave_power, shading='gouraud', cmap='viridis')

plt.colorbar(label='Mathematical Power')
plt.ylabel('Frequency')
plt.xlabel('Time (Months since 1961)')
plt.title('Wavelet Power Spectrum: East Atlantic Regime Shift')

# Limit the color scale to 80% of max to highlight the structural anomalies
plt.clim(0, np.max(wave_power) * 0.8) 
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pygam import LinearGAM, s, l

# ==============================================================================
# 1. PREPARE THE PYTHON ENVIRONMENT
# ==============================================================================
# Assume df_ortho is your pandas DataFrame. Drop any NaN rows first.
df_gam = df_ortho.dropna().copy()

# Define the feature matrix (X) and the target vector (y)
# Index 0: NAO_Lag0, Index 1: NAO_Lag135, Index 2: EA_Lag0
X = df_gam[['NAO_Lag0', 'NAO_Lag135', 'EA_Lag0']].values
y = df_gam['French_Temp'].values

# ==============================================================================
# 2. EXECUTE THE SPLINE PHYSICS
# ==============================================================================
# The GAM Architecture:
# s(0): Non-linear spline for instantaneous wind
# l(1): Strictly linear relationship for the oceanic gyre (stationary assumption)
# s(2): Non-linear spline for the East Atlantic pattern (to map the cold funnel)
master_gam = LinearGAM(s(0, n_splines=10) + l(1) + s(2, n_splines=10))
master_gam.gridsearch(X, y) # Autonomously finds the optimal smoothness penalty

# Output the statistical diagnostic summary
print(master_gam.summary())

# ==============================================================================
# 3. PLOT THE NON-LINEAR THRESHOLDS
# ==============================================================================
# We plot the partial dependence function for the East Atlantic parameter (feature index 2)
# to physically see the geometry of the atmospheric threshold.
XX = master_gam.generate_X_grid(term=2)
pdep, confi = master_gam.partial_dependence(term=2, X=XX, width=.95)

plt.figure(figsize=(8, 5))
# Plot the isolated effect
plt.plot(XX[:, 2], pdep, color='blue', label='Non-Linear Spline')
# Plot the 95% confidence intervals
plt.fill_between(XX[:, 2], confi[:, 0], confi[:, 1], color='lightblue', alpha=0.5)

plt.title('Non-Linear Threshold: East Atlantic Pattern (pyGAM)')
plt.xlabel('EA Z-Score')
plt.ylabel('Impact on French Temperature ($C_t$)')
plt.axhline(0, color='black', linestyle='--')
plt.legend()
plt.show()